# HealthSpark — Exploratory Data Analysis

This notebook explores the synthetic healthcare claims dataset to understand key patterns
in **denials, readmissions, costs, and provider performance** — the metrics that clinical
leadership and payer operations teams care most about.

**Dataset**: 500,000 synthetic claims across ~50,000 patients (HIPAA-safe)

---

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pyspark.sql import functions as F
from src.utils.spark_session import get_spark_session
from src.pipeline.ingestion import CLAIMS_SCHEMA, PATIENTS_SCHEMA

# Initialize Spark with Arrow optimization for fast toPandas() conversion
spark = get_spark_session(app_name='HealthSpark-EDA')

# Style settings
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print(f'Spark version: {spark.version}')

In [ ]:
# Load data with explicit schemas (same as the ETL pipeline)
claims_df = spark.read.option('header', 'true').option('dateFormat', 'yyyy-MM-dd').schema(CLAIMS_SCHEMA).csv('../data/raw/claims.csv')
patients_df = spark.read.option('header', 'true').schema(PATIENTS_SCHEMA).csv('../data/raw/patients.csv')

print(f'Claims: {claims_df.count():,} rows, {len(claims_df.columns)} columns')
print(f'Patients: {patients_df.count():,} rows, {len(patients_df.columns)} columns')
claims_df.printSchema()

## 1. Claim Denial Rate by Payer Type

**Business context**: Denial rate varies significantly by payer. Self-Pay claims have the highest
denial rates (~22%), while Medicare has the lowest (~8%). This pattern mirrors real-world data
where commercial and self-pay claims face more stringent utilization review.

In [ ]:
# Denial rate by payer — computed in Spark, plotted in matplotlib
denial_by_payer = (
    claims_df.groupBy('payer_type')
    .agg(
        F.avg('denial_flag').alias('denial_rate'),
        F.count('claim_id').alias('total_claims'),
    )
    .orderBy(F.col('denial_rate').desc())
    .toPandas()  # Convert to pandas for matplotlib (Arrow-optimized)
)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(denial_by_payer['payer_type'], denial_by_payer['denial_rate'] * 100, color=sns.color_palette('YlOrRd', len(denial_by_payer)))
ax.set_xlabel('Denial Rate (%)')
ax.set_title('Claim Denial Rate by Payer Type')
for bar, rate in zip(bars, denial_by_payer['denial_rate']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, f'{rate:.1%}', va='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 2. 30-Day Readmission Rate by Diagnosis

**Key finding**: Heart failure (I50.9) and COPD (J44.1) have the highest readmission rates,
consistent with CMS Hospital Readmissions Reduction Program (HRRP) penalties targeting these
specific conditions. Identifying high-risk patients at discharge is critical for reducing penalties.

In [ ]:
readmit_by_dx = (
    claims_df.groupBy('diagnosis_code')
    .agg(
        F.avg('readmission_30day').alias('readmit_rate'),
        F.count('claim_id').alias('claim_count'),
    )
    .where(F.col('claim_count') >= 1000)
    .orderBy(F.col('readmit_rate').desc())
    .toPandas()
)

fig, ax = plt.subplots(figsize=(12, 7))
colors = ['#d32f2f' if r > 0.20 else '#f57c00' if r > 0.15 else '#4caf50' for r in readmit_by_dx['readmit_rate']]
ax.barh(readmit_by_dx['diagnosis_code'], readmit_by_dx['readmit_rate'] * 100, color=colors)
ax.set_xlabel('30-Day Readmission Rate (%)')
ax.set_title('30-Day Readmission Rate by Diagnosis (ICD-10)')
ax.axvline(x=15, color='gray', linestyle='--', alpha=0.7, label='National avg ~15%')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Claim Cost Distribution

**Insight for leadership**: Healthcare costs follow a heavy right-skewed distribution —
a small percentage of claims account for a disproportionate share of total spend.
This is the "80/20 rule" of healthcare: ~20% of patients drive ~80% of costs.

In [ ]:
# Sample for plotting (full 500K is too many points for matplotlib)
cost_sample = claims_df.select('claim_amount', 'facility_type').sample(0.05, seed=42).toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram of claim amounts (log scale)
axes[0].hist(cost_sample['claim_amount'], bins=80, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Claim Amount ($)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Claim Amounts')
axes[0].set_xscale('log')
axes[0].axvline(cost_sample['claim_amount'].median(), color='red', linestyle='--', label=f'Median: ${cost_sample["claim_amount"].median():,.0f}')
axes[0].legend()

# Right: box plot by facility type
sns.boxplot(data=cost_sample, x='facility_type', y='claim_amount', ax=axes[1], showfliers=False)
axes[1].set_ylabel('Claim Amount ($)')
axes[1].set_title('Claim Cost by Facility Type')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 4. Provider Performance — Denial Rate Heatmap

**Operational insight**: Identifying high-denial providers by facility type helps
target quality improvement programs. Providers with above-average denial rates
may need coding education or utilization management support.

In [ ]:
# Top 20 providers by volume, cross-tabbed with facility type
provider_facility = (
    claims_df.groupBy('provider_id', 'facility_type')
    .agg(F.avg('denial_flag').alias('denial_rate'), F.count('*').alias('n'))
    .where(F.col('n') >= 50)
    .toPandas()
)

# Pivot for heatmap — top 15 providers by total claims
top_providers = provider_facility.groupby('provider_id')['n'].sum().nlargest(15).index
heatmap_data = provider_facility[provider_facility['provider_id'].isin(top_providers)].pivot_table(
    index='provider_id', columns='facility_type', values='denial_rate', aggfunc='mean'
).fillna(0)

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(heatmap_data * 100, annot=True, fmt='.1f', cmap='RdYlGn_r', ax=ax,
            cbar_kws={'label': 'Denial Rate (%)'})
ax.set_title('Provider Denial Rate (%) by Facility Type — Top 15 Providers')
ax.set_ylabel('Provider ID')
plt.tight_layout()
plt.show()

## 5. Age vs Length of Stay

**Clinical insight**: Length of stay increases with age, particularly after 65.
Patients with multiple comorbidities (color intensity) have even longer stays.
This informs discharge planning and post-acute care resource allocation.

In [ ]:
age_los = claims_df.select('age', 'length_of_stay', 'comorbidity_count', 'readmission_30day').sample(0.02, seed=42).toPandas()

fig, ax = plt.subplots(figsize=(12, 6))
scatter = ax.scatter(
    age_los['age'], age_los['length_of_stay'],
    c=age_los['comorbidity_count'], cmap='YlOrRd',
    alpha=0.4, s=15, edgecolors='none'
)
plt.colorbar(scatter, label='Comorbidity Count')
ax.set_xlabel('Age')
ax.set_ylabel('Length of Stay (days)')
ax.set_title('Age vs Length of Stay (colored by comorbidity count)')

# Add trend line
z = np.polyfit(age_los['age'], age_los['length_of_stay'], 1)
p = np.poly1d(z)
ax.plot(sorted(age_los['age'].unique()), p(sorted(age_los['age'].unique())), 'b--', alpha=0.8, linewidth=2, label='Trend')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Monthly Claim Volume & Cost Trends

**Trend analysis**: Understanding seasonal patterns in claim volume and costs
helps with capacity planning and budget forecasting.

In [ ]:
monthly_trends = (
    claims_df
    .withColumn('month', F.date_trunc('month', 'admit_date'))
    .groupBy('month')
    .agg(
        F.count('claim_id').alias('claim_volume'),
        F.sum('claim_amount').alias('total_cost'),
        F.avg('denial_flag').alias('denial_rate'),
    )
    .orderBy('month')
    .toPandas()
)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(monthly_trends['month'], monthly_trends['claim_volume'], color='steelblue', linewidth=2)
axes[0].fill_between(monthly_trends['month'], monthly_trends['claim_volume'], alpha=0.2, color='steelblue')
axes[0].set_ylabel('Claim Volume')
axes[0].set_title('Monthly Claim Volume (2022-2024)')

axes[1].plot(monthly_trends['month'], monthly_trends['total_cost'] / 1e6, color='#d32f2f', linewidth=2)
axes[1].fill_between(monthly_trends['month'], monthly_trends['total_cost'] / 1e6, alpha=0.2, color='#d32f2f')
axes[1].set_ylabel('Total Cost ($M)')
axes[1].set_xlabel('Month')
axes[1].set_title('Monthly Total Claim Cost (2022-2024)')

plt.tight_layout()
plt.show()

## 7. Insurance Type Distribution

**Payer mix**: Understanding the distribution of insurance types helps forecast
reimbursement rates and manage payer-specific workflows.

In [ ]:
insurance_dist = claims_df.groupBy('insurance_type').count().orderBy(F.col('count').desc()).toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
axes[0].pie(insurance_dist['count'], labels=insurance_dist['insurance_type'], autopct='%1.1f%%',
            startangle=90, colors=sns.color_palette('Set2', len(insurance_dist)))
axes[0].set_title('Claims by Insurance Type')

# Bar chart with denial overlay
ins_denial = (
    claims_df.groupBy('insurance_type')
    .agg(F.avg('denial_flag').alias('denial_rate'), F.avg('claim_amount').alias('avg_cost'))
    .toPandas()
)
x = range(len(ins_denial))
bars = axes[1].bar(x, ins_denial['avg_cost'], color=sns.color_palette('Set2', len(ins_denial)), alpha=0.8)
ax2 = axes[1].twinx()
ax2.plot(x, ins_denial['denial_rate'] * 100, 'ro-', linewidth=2, markersize=8)
ax2.set_ylabel('Denial Rate (%)', color='red')
axes[1].set_xticks(x)
axes[1].set_xticklabels(ins_denial['insurance_type'], rotation=35, ha='right')
axes[1].set_ylabel('Avg Claim Amount ($)')
axes[1].set_title('Avg Cost & Denial Rate by Insurance Type')

plt.tight_layout()
plt.show()

## 8. Readmission Risk Factors — Correlation Analysis

**For the predictive model**: Understanding which features correlate most strongly
with 30-day readmission helps prioritize feature engineering and interpret model results.

In [ ]:
# Correlation matrix for numeric features
numeric_cols = ['readmission_30day', 'age', 'length_of_stay', 'claim_amount', 'paid_amount',
                'comorbidity_count', 'denial_flag']
corr_data = claims_df.select(numeric_cols).sample(0.1, seed=42).toPandas()

fig, ax = plt.subplots(figsize=(10, 8))
corr_matrix = corr_data.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Correlation Matrix — Key Numeric Features')
plt.tight_layout()
plt.show()

In [ ]:
# Clean up
spark.stop()
print('SparkSession stopped.')

---

## Key Takeaways for Clinical Leadership

1. **Denial rates** are highest for Self-Pay (22%) — consider pre-authorization workflows
2. **Heart failure and COPD** drive the highest readmission rates — target transitional care programs
3. **Cost distribution** is heavily right-skewed — a small fraction of claims drives most spend
4. **Provider variation** in denial rates suggests opportunities for coding improvement
5. **Age and comorbidities** are the strongest demographic predictors of longer stays
6. **Monthly trends** show stable volume with predictable seasonal patterns

These insights feed directly into the predictive model in the next notebook.